In [36]:
# ── CELL 12A — Tendance STL → Rééchantillonnage Trimestriel ────────────────
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import linregress

YEARS_MONTHLY = list(range(2016, 2025))
full_idx = pd.date_range(f'{YEARS_MONTHLY[0]}-01-01',
                          f'{YEARS_MONTHLY[-1]}-12-01', freq='MS')

records_q = []
for zone in ZONE_ORDER:
    res = stl_results[zone]
    ts_trend = pd.Series(res.trend, index=full_idx, name='trend_vv')
    ts_q = ts_trend.resample('QS').mean()
    for date, vv_trend in ts_q.items():
        if np.isnan(vv_trend):
            continue
        quarter = (date.month - 1) // 3 + 1
        records_q.append({
            'zone'     : zone,
            'date'     : date,
            'year'     : date.year,
            'quarter'  : quarter,
            'label'    : f'{date.year}-Q{quarter}',
            'vv_trend' : round(float(vv_trend), 4),
        })

df_q = pd.DataFrame(records_q)

# ── Calibration zone-par-zone (régression VV_trend_Q3 ~ UF_annuel) ──────────
# CORRECTION : la calibration globale VV_ZERO=-14→UF=0% était fausse.
# La tendance STL = VV moyen zone (mélange bâti+sable), pas pixel pur.
# Une zone à 50% bâti a VV moyen ≈ -14 dB → mappée à 0% avec l'ancienne formule.
# Fix : régression linéaire par zone calée sur les UF annuelles connues.

UF_ANNUAL = {
    'Arafat':       [95.5,95.1,95.3,95.9,96.0,96.2,97.2,97.3,97.1],
    'Dar Naim':     [71.5,64.4,63.8,65.9,66.5,67.9,71.1,74.7,77.5],
    'El Mina':      [42.9,41.3,44.3,45.9,47.8,47.4,48.5,48.7,50.3],
    'Ksar':         [33.9,34.3,35.0,37.8,39.5,41.7,44.5,47.2,50.7],
    'Riyadh':       [31.2,30.5,30.8,32.1,32.7,33.6,35.4,36.6,40.8],
    'Sebkha':       [52.8,53.5,54.6,57.4,60.0,59.7,63.7,64.2,66.1],
    'Tevragh Zein': [48.3,47.8,47.4,51.4,53.5,54.0,54.6,55.8,58.1],
    'Teyarett':     [32.8,33.0,33.8,36.8,38.7,40.8,42.5,45.0,47.7],
    'Toujounine':   [53.2,53.6,54.6,56.5,57.4,58.1,59.8,61.4,62.0],
}

zone_calib = {}
print('Calibration zone-par-zone (VV_trend_Q3 ~ UF_annuel) :')
print(f'{"Zone":20s} {"a":>8s} {"b":>10s} {"r²":>6s}')
print('-'*48)
for zone in ZONE_ORDER:
    sub_q3 = (df_q[df_q['zone']==zone]
              .query('quarter==3')
              .sort_values('year'))
    vv = sub_q3['vv_trend'].values
    uf = np.array(UF_ANNUAL[zone])
    n  = min(len(vv), len(uf))
    slope, intercept, r, _, _ = linregress(vv[:n], uf[:n])
    zone_calib[zone] = (slope, intercept)
    print(f'{zone:20s} {slope:>8.4f} {intercept:>10.4f} {r**2:>6.3f}')

df_q['uf_trend'] = df_q.apply(
    lambda row: float(np.clip(
        zone_calib[row['zone']][0] * row['vv_trend'] + zone_calib[row['zone']][1],
        0, 100
    )), axis=1
)

# Vérification cohérence
print('\nVérification cohérence UF trimestrielle vs UF annuelle :')
print(f'{"Zone":20s} {"UF_annuel_2024":>15s} {"UF_trim_2024":>14s} {"Écart":>8s}')
print('-'*62)
for zone in ZONE_ORDER:
    sub_q   = df_q[(df_q['zone']==zone) & (df_q['year']==2024)]
    uf_q_24 = sub_q['uf_trend'].mean() if len(sub_q)>0 else np.nan
    uf_a_24 = UF_ANNUAL[zone][-1]
    ecart   = uf_q_24 - uf_a_24 if not np.isnan(uf_q_24) else np.nan
    print(f'{zone:20s} {uf_a_24:>15.1f} {uf_q_24:>14.1f} {ecart:>+8.1f}')

df_q.to_csv('/outputs/quarterly_uf_stl.csv', index=False)
print(f'\n✓ {len(df_q)} enregistrements trimestriels')
print(f'  Zones : {df_q["zone"].nunique()}')
print(f'  Période : {df_q["label"].iloc[0]} → {df_q["label"].iloc[-1]}')
print('✓ Saved: /outputs/quarterly_uf_stl.csv')

Calibration zone-par-zone (VV_trend_Q3 ~ UF_annuel) :
Zone                        a          b     r²
------------------------------------------------
Arafat                 3.5163   125.6275  0.876
Dar Naim               9.0418   176.5272  0.952
Toujounine             8.5129   168.5925  0.919
Ksar                   6.4622   139.7591  0.856
Tevragh Zein           7.3984   157.6100  0.695
Sebkha                28.4167   494.9795  0.695
Teyarett               7.1608   150.6123  0.870
El Mina               12.3247   230.5909  0.851
Riyadh                 3.4999    87.1397  0.413

Vérification cohérence UF trimestrielle vs UF annuelle :
Zone                  UF_annuel_2024   UF_trim_2024    Écart
--------------------------------------------------------------
Arafat                          97.1           97.1     +0.0
Dar Naim                        77.5           76.8     -0.7
Toujounine                      62.0           62.2     +0.2
Ksar                            50.7           50.8 

In [37]:
# ── CELL 12B — Visualisation Séries Trimestrielles ──────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(22, 16))
fig.suptitle('Indice Urbain Trimestriel Désaisonnalisé — Nouakchott 2016-2024\n'
             'Basé sur la composante Tendance STL  |  Saisonnalité éliminée',
             fontsize=14, fontweight='bold')

for ax, zone in zip(axes.flat, ZONE_ORDER):
    sub   = df_q[df_q['zone']==zone].sort_values('date')
    color = ZONE_COLORS[zone]
    dates = pd.to_datetime(sub['date'])

    ax.plot(dates, sub['uf_trend'], color=color, lw=2.5,
            marker='o', ms=4, zorder=3, label='UF trimestrielle')

    rolling_std = sub['uf_trend'].rolling(4, center=True, min_periods=2).std()
    ax.fill_between(dates,
                    sub['uf_trend'] - rolling_std,
                    sub['uf_trend'] + rolling_std,
                    color=color, alpha=0.12, zorder=1, label='±1σ glissant')

    # Diamants annuels — indice annuel de référence
    for i, year in enumerate(YEARS_MONTHLY):
        ax.scatter(pd.Timestamp(f'{year}-07-01'), UF_ANNUAL[zone][i],
                   marker='D', s=40, color='black', zorder=5, alpha=0.6)

    # Régression
    x_num = np.arange(len(sub))
    slope, intercept, r, p, _ = stats.linregress(x_num, sub['uf_trend'])
    trend_line = intercept + slope * x_num
    ax.plot(dates, trend_line, color=color, lw=1.5, ls='--', alpha=0.7, zorder=2)

    slope_yr = slope * 4
    sig = '✓' if p < 0.05 else 'ns'
    ax.text(0.03, 0.06,
            f'+{slope_yr:.2f} pp/an  (p={p:.3f} {sig})',
            transform=ax.transAxes, fontsize=8, color=color,
            bbox=dict(facecolor='white', alpha=0.75, boxstyle='round', linewidth=0))

    ax.set_title(zone, fontsize=10, fontweight='bold', color=color)
    ax.set_ylabel('Fraction Urbaine (%)', fontsize=8)
    ax.set_ylim(bottom=max(0, sub['uf_trend'].min()-5))
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.2, ls=':')
    ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=7, rotation=45)

axes[0,0].scatter([], [], marker='D', s=40, color='black', alpha=0.6,
                  label='Indice annuel (référence)')
axes[0,0].legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig('/outputs/quarterly_01_series.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: quarterly_01_series.png')

✓ Saved: quarterly_01_series.png


In [38]:
# ── CELL 12C — Heatmap Trimestrielle et Accélération de Croissance ─────────
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle('Indice Trimestriel — Heatmap et Accélération de Croissance\n'
             'Nouakchott 2016-2024  |  Désaisonnalisé STL',
             fontsize=13, fontweight='bold')

ax = axes[0]
pivot_q = df_q.pivot_table(index='zone', columns='label',
                            values='uf_trend', aggfunc='mean')
zone_order_uf = sorted(ZONE_ORDER, key=lambda z: UF_ANNUAL[z][-1], reverse=True)
pivot_q = pivot_q.reindex(zone_order_uf)

cmap_hm = LinearSegmentedColormap.from_list('uf_hm',
    ['#f0f4f8','#c9d6e3','#7fb3d3','#2c7bb6','#1a3a5c'])
im = ax.imshow(pivot_q.values, aspect='auto', cmap=cmap_hm,
               vmin=0, vmax=100, interpolation='nearest')

for i in range(len(pivot_q.index)):
    for j in range(len(pivot_q.columns)):
        if j % 2 == 0:
            val = pivot_q.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                        fontsize=5.5,
                        color='white' if val > 50 else 'black')

ax.set_yticks(range(len(zone_order_uf)))
ax.set_yticklabels(zone_order_uf, fontsize=8)
q_labels  = list(pivot_q.columns)
q1_idx    = [i for i, l in enumerate(q_labels) if 'Q1' in l]
q1_labels = [l.replace('-Q1','') for l in q_labels if 'Q1' in l]
ax.set_xticks(q1_idx)
ax.set_xticklabels(q1_labels, fontsize=8, rotation=45)
ax.set_title('Fraction Urbaine % par Zone et Trimestre', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7, label='UF (%)')

ax2 = axes[1]
periods = [
    ('2016-2018', 2016, 2018, '#457b9d'),
    ('2019-2021', 2019, 2021, '#2a9d8f'),
    ('2022-2024', 2022, 2024, '#e63946'),
]
slopes_by_period = {z: {} for z in ZONE_ORDER}
for label_p, y1, y2, color_p in periods:
    for zone in ZONE_ORDER:
        sub = df_q[(df_q['zone']==zone) &
                   (df_q['year']>=y1) & (df_q['year']<=y2)].sort_values('date')
        if len(sub) < 4:
            continue
        x = np.arange(len(sub))
        slope, _, _, p, _ = stats.linregress(x, sub['uf_trend'])
        slopes_by_period[zone][label_p] = slope * 4

x_pos  = np.arange(len(ZONE_ORDER))
width  = 0.25
for i, (label_p, y1, y2, color_p) in enumerate(periods):
    vals = [slopes_by_period[z].get(label_p, 0) for z in ZONE_ORDER]
    ax2.bar(x_pos + i*width, vals, width, label=label_p,
            color=color_p, alpha=0.85, edgecolor='white')

ax2.axhline(0, color='black', lw=1)
ax2.set_xticks(x_pos + width)
ax2.set_xticklabels(ZONE_ORDER, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Vitesse de croissance (pp/an)', fontsize=10)
ax2.set_title('Accélération / Décélération par Période\n'
              '(pente de régression sur sous-période)', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.25, ls=':')

fastest = max(ZONE_ORDER,
              key=lambda z: slopes_by_period[z].get('2022-2024', -999))
ax2.text(0.98, 0.97,
         f'Croissance la plus rapide\n2022-2024 : {fastest}',
         transform=ax2.transAxes, ha='right', va='top', fontsize=9,
         bbox=dict(facecolor='#e63946', alpha=0.15, boxstyle='round'))

plt.tight_layout()
plt.savefig('/outputs/quarterly_02_heatmap_acceleration.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: quarterly_02_heatmap_acceleration.png')

print()
print('Vitesses de croissance (pp/an) par sous-période :')
print(f'{"Zone":20s} {"2016-18":>10s} {"2019-21":>10s} {"2022-24":>10s} {"Accélération":>14s}')
print('-'*58)
for zone in ZONE_ORDER:
    s1 = slopes_by_period[zone].get('2016-2018', np.nan)
    s2 = slopes_by_period[zone].get('2019-2021', np.nan)
    s3 = slopes_by_period[zone].get('2022-2024', np.nan)
    acc  = s3 - s1 if not np.isnan(s1) and not np.isnan(s3) else np.nan
    flag = '↑ accélère' if acc > 0.3 else ('↓ ralentit' if acc < -0.3 else '→ stable')
    print(f'{zone:20s} {s1:>+10.2f} {s2:>+10.2f} {s3:>+10.2f} {flag}')

✓ Saved: quarterly_02_heatmap_acceleration.png

Vitesses de croissance (pp/an) par sous-période :
Zone                    2016-18    2019-21    2022-24   Accélération
----------------------------------------------------------
Arafat                    +0.19      +0.24      +0.14 → stable
Dar Naim                  -3.04      +1.50      +2.57 ↑ accélère
Toujounine                -0.08      +0.64      +1.59 ↑ accélère
Ksar                      -2.03      +1.72      +4.31 ↑ accélère
Tevragh Zein              -1.24      +0.59      +2.76 ↑ accélère
Sebkha                    +0.95      +0.65      +2.87 ↑ accélère
Teyarett                  -1.66      +1.61      +3.71 ↑ accélère
El Mina                   +1.50      +1.48      +0.85 ↓ ralentit
Riyadh                    -1.72      +0.54      +1.96 ↑ accélère


In [39]:
# ── CELL 12D — Mann-Kendall (implémentation manuelle — pas de dépendance externe) ──
from scipy.stats import norm as _norm

def mann_kendall_test(x):
    x = np.array(x, dtype=float)
    n = len(x)
    s = sum(np.sign(x[j] - x[k]) for k in range(n-1) for j in range(k+1, n))
    var_s = n*(n-1)*(2*n+5)/18
    z = (s-1)/np.sqrt(var_s) if s>0 else (s+1)/np.sqrt(var_s) if s<0 else 0.0
    p = 2*(1 - _norm.cdf(abs(z)))
    tau = s / (0.5*n*(n-1))
    slopes = [(x[j]-x[k])/(j-k) for k in range(n-1) for j in range(k+1, n)]
    slope  = np.median(slopes)
    trend  = ('increasing' if p<0.05 and s>0
              else 'decreasing' if p<0.05 and s<0
              else 'no trend')
    class Result:
        pass
    r = Result()
    r.Tau=tau; r.p=p; r.s=s; r.slope=slope; r.trend=trend
    return r

print('Test de Mann-Kendall sur les séries trimestrielles désaisonnalisées')
print('H0 : pas de tendance monotone')
print('='*72)
print(f'{"Zone":20s} {"stat Tau":>10s} {"p-value":>10s} {"Tendance":>12s} {"slope":>10s} {"sig":>6s}')
print('-'*72)

mk_results = {}
for zone in ZONE_ORDER:
    sub = df_q[df_q['zone']==zone].sort_values('date')['uf_trend'].dropna()
    res = mann_kendall_test(sub)
    slope_q = res.slope * 4   # trimestre → annuel
    mk_results[zone] = {
        'tau': res.Tau, 'p': res.p, 'trend': res.trend,
        'slope': slope_q, 's': res.s
    }
    sig = ('***' if res.p<0.001 else '**' if res.p<0.01
           else '*' if res.p<0.05 else 'ns')
    print(f'{zone:20s} {res.Tau:>+10.3f} {res.p:>10.4f} '
          f'{res.trend:>12s} {slope_q:>+10.3f} {sig:>6s}')

print()
sig_zones = [z for z in ZONE_ORDER if mk_results[z]['p'] < 0.05]
print(f'  {len(sig_zones)}/9 zones significatives (p<0.05)')
for z in sig_zones:
    print(f'  ✓ {z}: {mk_results[z]["slope"]:+.2f} pp/an (p={mk_results[z]["p"]:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Test de Mann-Kendall — Tendance Trimestrielle Désaisonnalisée\n'
             'Nouakchott 2016-2024  |  Toutes zones', fontsize=13, fontweight='bold')

ax = axes[0]
taus  = [mk_results[z]['tau'] for z in ZONE_ORDER]
pvals = [mk_results[z]['p']   for z in ZONE_ORDER]
cols  = [ZONE_COLORS[z] if p<0.05 else '#adb5bd' for z,p in zip(ZONE_ORDER, pvals)]
bars  = ax.barh(ZONE_ORDER, taus, color=cols, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Tau de Kendall  [+1 = tendance croissante parfaite]')
ax.set_title('Tau de Kendall par Moughataa', fontweight='bold')
for bar, p in zip(bars, pvals):
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
    if sig:
        ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                sig, va='center', fontsize=10, color='#2a9d8f', fontweight='bold')
ax.grid(axis='x', alpha=0.3, ls=':')

ax2 = axes[1]
slopes = [mk_results[z]['slope'] for z in ZONE_ORDER]
cols2  = [ZONE_COLORS[z] for z in ZONE_ORDER]
ax2.barh(ZONE_ORDER, slopes, color=cols2, alpha=0.85, edgecolor='white')
ax2.axvline(0, color='black', lw=1)
ax2.set_xlabel('Vitesse de croissance urbaine (pp/an)')
ax2.set_title('Pente de Theil-Sen (pp/an)', fontweight='bold')
ax2.grid(axis='x', alpha=0.3, ls=':')

plt.tight_layout()
plt.savefig('/outputs/quarterly_03_mann_kendall.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: quarterly_03_mann_kendall.png')

Test de Mann-Kendall sur les séries trimestrielles désaisonnalisées
H0 : pas de tendance monotone
Zone                   stat Tau    p-value     Tendance      slope    sig
------------------------------------------------------------------------
Arafat                   +0.949     0.0000   increasing     +0.276    ***
Dar Naim                 +0.644     0.0000   increasing     +1.789    ***
Toujounine               +0.810     0.0000   increasing     +0.963    ***
Ksar                     +0.644     0.0000   increasing     +1.713    ***
Tevragh Zein             +0.594     0.0000   increasing     +0.846    ***
Sebkha                   +0.698     0.0000   increasing     +1.328    ***
Teyarett                 +0.708     0.0000   increasing     +1.697    ***
El Mina                  +0.733     0.0000   increasing     +0.938    ***
Riyadh                   +0.197     0.0939     no trend     +0.284     ns

  8/9 zones significatives (p<0.05)
  ✓ Arafat: +0.28 pp/an (p=0.0000)
  ✓ Dar Naim: +1.

In [40]:
# ── CELL 12E — Indice Ville Trimestriel + Figure Synthèse ───────────────────

city_q_records = []
for label_q in sorted(df_q['label'].unique()):
    sub = df_q[df_q['label']==label_q]
    total_w = 0; weighted_uf = 0
    for zone in ZONE_ORDER:
        row = sub[sub['zone']==zone]
        if len(row)==0: continue
        val = row['uf_trend'].values[0]
        if np.isnan(val): continue
        w = areas[zone]
        weighted_uf += val * w
        total_w     += w
    if total_w > 0:
        date_q = pd.to_datetime(df_q[df_q['label']==label_q]['date'].iloc[0])
        city_q_records.append({
            'label'  : label_q,
            'date'   : date_q,
            'year'   : date_q.year,
            'quarter': (date_q.month-1)//3+1,
            'uf_city': round(weighted_uf/total_w, 3)
        })

df_city_q = pd.DataFrame(city_q_records).sort_values('date')

res_city     = mann_kendall_test(df_city_q['uf_city'].dropna())
slope_city_q = res_city.slope * 4

print(f'Indice ville trimestriel :')
print(f'  Mann-Kendall Tau = {res_city.Tau:+.3f}  p = {res_city.p:.4f}  '
      f'tendance = {res_city.trend}')
print(f'  Pente Theil-Sen  = {slope_city_q:+.3f} pp/an')

fig, axes = plt.subplots(2, 1, figsize=(18, 14),
                          gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle('Indice Urbain Trimestriel Désaisonnalisé — Synthèse Finale\n'
             'Nouakchott 2016-2024  |  Basé sur Tendance STL Sentinel-1 SAR',
             fontsize=14, fontweight='bold')

ax = axes[0]
for zone in ZONE_ORDER:
    sub   = df_q[df_q['zone']==zone].sort_values('date')
    dates = pd.to_datetime(sub['date'])
    color = ZONE_COLORS[zone]
    ax.plot(dates, sub['uf_trend'], color=color, lw=1.8, alpha=0.85, label=zone)
    last = sub.iloc[-1]
    zone_label = f"{zone.split()[0]}\n{last['uf_trend']:.0f}%"
    ax.annotate(zone_label,
                (pd.to_datetime(last['date']), last['uf_trend']),
                xytext=(8, 0), textcoords='offset points',
                fontsize=6.5, color=color, va='center')

ax.set_ylabel('Fraction Urbaine Estimée (%)', fontsize=11)
ax.set_title('Trajectoires Trimestrielles par Moughataa', fontweight='bold')
ax.legend(fontsize=8, loc='upper left', ncol=3)
ax.grid(alpha=0.2, ls=':')
ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
ax.tick_params(axis='x', labelsize=9)
for year in YEARS_MONTHLY:
    ax.axvline(pd.Timestamp(f'{year}-01-01'), color='grey',
               lw=0.5, ls=':', alpha=0.4)

ax2 = axes[1]
ax2.fill_between(df_city_q['date'], df_city_q['uf_city'],
                 alpha=0.15, color='#1d3557')
ax2.plot(df_city_q['date'], df_city_q['uf_city'],
         color='#1d3557', lw=3, marker='o', ms=4, label='Indice ville')

x_city = np.arange(len(df_city_q))
s_c, i_c, _, _, _ = stats.linregress(x_city, df_city_q['uf_city'])
ax2.plot(df_city_q['date'], i_c + s_c*x_city,
         color='#e63946', lw=2, ls='--',
         label=f'Tendance ({s_c*4:+.2f} pp/an)')

for year in YEARS_MONTHLY:
    sub_y = df_city_q[df_city_q['year']==year]
    if len(sub_y) > 0:
        mean_y = sub_y['uf_city'].mean()
        ax2.annotate(f'{mean_y:.1f}%',
                     (pd.Timestamp(f'{year}-07-01'), mean_y),
                     xytext=(0, 8), textcoords='offset points',
                     fontsize=7.5, ha='center', color='#1d3557')

ax2.set_ylabel('Fraction Urbaine Ville (%)', fontsize=11)
ax2.set_title(f'Indice Ville Global — Mann-Kendall : Tau={res_city.Tau:+.3f}  '
              f'p={res_city.p:.4f}  ({res_city.trend})', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.2, ls=':')
ax2.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
ax2.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
ax2.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('/outputs/quarterly_04_final_synthesis.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: quarterly_04_final_synthesis.png')

print()
print('=== RÉCAPITULATIF PARTIE 12 ===')
print(f'  Indice trimestriel : {len(df_city_q)} points ville  |  {len(df_q)} points zone')
print(f'  Tendance ville     : {slope_city_q:+.3f} pp/an  (p={res_city.p:.4f}, {res_city.trend})')
for f in ['quarterly_01_series.png', 'quarterly_02_heatmap_acceleration.png',
          'quarterly_03_mann_kendall.png', 'quarterly_04_final_synthesis.png',
          'quarterly_uf_stl.csv']:
    print(f'  ✓ /outputs/{f}')

Indice ville trimestriel :
  Mann-Kendall Tau = +0.733  p = 0.0000  tendance = increasing
  Pente Theil-Sen  = +0.969 pp/an
✓ Saved: quarterly_04_final_synthesis.png

=== RÉCAPITULATIF PARTIE 12 ===
  Indice trimestriel : 36 points ville  |  324 points zone
  Tendance ville     : +0.969 pp/an  (p=0.0000, increasing)
  ✓ /outputs/quarterly_01_series.png
  ✓ /outputs/quarterly_02_heatmap_acceleration.png
  ✓ /outputs/quarterly_03_mann_kendall.png
  ✓ /outputs/quarterly_04_final_synthesis.png
  ✓ /outputs/quarterly_uf_stl.csv
